# MGMT298D: Science and Strategy of AI
### Week 1 - Linear Regression with Feature Engineering
### Application: Demand Forecasting at H&M


**Student Name:**

## Import Libraries and Data

In [ ]:
# Import necessary libraries for data manipulation, machine learning, and metrics.
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

df = pd.read_csv("https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/refs/heads/main/HMData.csv")

# AVAILABLE CATEGORIES (product_type_name):
# Bag, Ballerinas, Belt, Bikini top, Blazer, Blouse, Bodysuit, Boots,
# Bra, Bra extender, Braces, Cardigan, Coat, Costumes, Dress,
# Dungarees, Earring, Felt hat, Hat/beanie, Hat/brim, Hoodie, Jacket,
# Jumpsuit/Playsuit, Kids Underwear top, Leggings/Tights, Night gown,
# Nipple covers, Polo shirt, Pyjama bottom, Sandals, Shirt, Shorts,
# Skirt, Sneakers, Socks, Sunglasses, Sweater, Swimsuit, Swimwear bottom,
# T-shirt, Top, Trousers, Underwear body, Underwear bottom,
# Underwear Tights, Unknown, Vest top

SELECTED_PRODUCT = "Dress"
df_sub = df[df["name"] == SELECTED_PRODUCT].copy()

## Feature Engineering

In [ ]:
# --- Feature engineering (fixed) ---
# Creates temporal features (lags, rolling averages) and splits data into Train/Test sets based on time.

# Assuming month columns exist as binary flags (e.g., 'January', 'February', ..., 'December')
# and a row corresponds to a specific month. This section creates 'month_num' if it's missing.
month_cols = ["January", "February", "March", "April", "May", "June",
              "July", "August", "September", "October", "November", "December"]

# Create 'month_num' column if it doesn't exist by finding the active month column.
# Ensure all expected month columns are present before attempting idxmax
if all(col in df_sub.columns for col in month_cols):
    df_sub["month_num"] = df_sub[month_cols].idxmax(axis=1).apply(lambda x: month_cols.index(x) + 1)
else:
    raise KeyError("One or more expected month columns (e.g., 'January', 'February') are missing from the DataFrame. Unable to create 'month_num'.")

# lags
for i in range(1, 4):
    df_sub[f"lag_m{i}"] = df_sub.groupby(["id"])["demand"].shift(i)

# rolling mean of previous 3 months (lookback window)
df_sub["ma_3"] = (
    df_sub
    .groupby(["id"], group_keys=False)
    .apply(lambda g: g["demand"].shift(1).rolling(3, min_periods=1).mean(), include_groups=False)
)

# price change within article+channel
df_sub["price_change"] = (
    df_sub.groupby(["id"])["price"].pct_change()
)

# fill engineered NaNs
fe_cols = ["lag_m1", "lag_m2", "lag_m3", "ma_3", "price_change"]
df_sub[fe_cols] = df_sub[fe_cols].fillna(0)

# time-based train/test split
months_sub = np.sort(df_sub["month_num"].dropna().unique())
train = df_sub[df_sub["month_num"] < months_sub[-1]].copy()
test  = df_sub[df_sub["month_num"] == months_sub[-1]].copy()

print("[INFO] train rows:", len(train), "| test rows:", len(test))

[INFO] train rows: 30794 | test rows: 2777


## Model Training and Evaluation

In [ ]:
# =========================================
# Fit 3 LASSO Models (evaluate using MAE)
# =========================================
# Trains three models with increasing complexity:
# Model A: Baseline (Price + Category)
# Model B: Temporal Features added
# Model C: Pairwise Interactions added

# Define month columns globally or ensure accessibility
month_cols = ["January", "February", "March", "April", "May", "June",
              "July", "August", "September", "October", "November", "December"]

# Get a list of actual month columns present in the DataFrame after any filtering
actual_month_cols = [col for col in month_cols if col in train.columns]

# ---------- Model A: baseline (price + month) ----------
modelA_num = ["price"]
# Directly use the binary month columns, no get_dummies needed for them
train_A = train[modelA_num + actual_month_cols].copy()
test_A  = test[modelA_num + actual_month_cols].copy()
train_A, test_A = train_A.align(test_A, join="left", axis=1, fill_value=0)

scaler = StandardScaler()
train_A[modelA_num] = scaler.fit_transform(train_A[modelA_num])
test_A[modelA_num]  = scaler.transform(test_A[modelA_num])

X_tr, X_te = train_A.values, test_A.values
y_tr, y_te = train["demand"], test["demand"]

lasso_A = LassoCV(cv=5, random_state=0).fit(X_tr, y_tr)
pred_A = lasso_A.predict(X_te)

mae_A = mean_absolute_error(y_te, pred_A)
print(f"Model A — MAE={mae_A:.2f}")


# ---------- Model B: add temporal features (lags, MA, price change) ----------
modelB_num = ["price", "price_change", "lag_m1", "lag_m2", "lag_m3", "ma_3"]
# Directly use the binary month columns, no get_dummies needed for them
train_B = train[modelB_num + actual_month_cols].copy()
test_B  = test[modelB_num + actual_month_cols].copy()
train_B, test_B = train_B.align(test_B, join="left", axis=1, fill_value=0)

scaler = StandardScaler()
train_B[modelB_num] = scaler.fit_transform(train_B[modelB_num])
test_B[modelB_num]  = scaler.transform(test_B[modelB_num])

X_tr, X_te = train_B.values, test_B.values

lasso_B = LassoCV(cv=5, random_state=0).fit(X_tr, y_tr)
pred_B = lasso_B.predict(X_te)

mae_B = mean_absolute_error(y_te, pred_B)
print(f"Model B — MAE={mae_B:.2f}")


# ---------- Model C: include pairwise interactions ----------
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_tr = poly.fit_transform(train_B[modelB_num])
poly_te = poly.transform(test_B[modelB_num])

# For Model C, we need to ensure the columns used for interaction match those dropped.
# The `drop` method will handle modelB_num from train_B, leaving actual_month_cols and then concatenate with poly_tr/poly_te
X_tr_C = np.hstack([train_B[actual_month_cols].values, poly_tr])
X_te_C = np.hstack([test_B[actual_month_cols].values, poly_te])

lasso_C = LassoCV(cv=5, random_state=0).fit(X_tr_C, y_tr)
pred_C = lasso_C.predict(X_te_C)

mae_C = mean_absolute_error(y_te, pred_C)
print(f"Model C — MAE={mae_C:.2f}")

Model A — MAE=44.07
Model B — MAE=31.97
Model C — MAE=30.71
